# Multi-stage validation and failure analysis

The run recorded Fast, Anti-Mirage and Full measurements. Gate decisions and per-candidate rejection reasons are missing. The run reported zero validated exports. No historical Holdout table was recovered.

In [ ]:
from pathlib import Path
import csv
import json
from collections import Counter, defaultdict
import matplotlib.pyplot as plt

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "results/phase3/EXPERIMENT_MANIFEST.json").is_file())
results = root / "results"

def load_csv(relative):
    with (results / relative).open(encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))


In [ ]:
evaluations = load_csv("phase3/beast_stage_results.csv")
stages = ["fast", "anti-mirage", "full"]
observations = Counter(r["stage"] for r in evaluations)
stages = [stage for stage in stages if observations[stage]]
fig, (left, right) = plt.subplots(1, 2, figsize=(10, 3.6))
left.bar(stages, [observations[s] for s in stages])
left.set(ylabel="Recorded candidate evaluations")
for ix, stage in enumerate(stages):
    values = [float(r["combined"]) for r in evaluations if r["stage"] == stage and r["combined"]]
    right.scatter([ix]*len(values), values, alpha=0.75)
right.set(xticks=range(len(stages)), xticklabels=stages, ylabel="Combined reward delta")
right.axhline(0, linewidth=0.8, color="grey")
plt.tight_layout()
plt.show()

In [ ]:
replays = load_csv("phase3/beast_replay_statistics.csv")
print(Counter(r["mode"] for r in replays))
print("Rows with a recorded first lead-loss step:", sum(bool(r["first_lead_loss_step"]) for r in replays))
full = [r for r in evaluations if r["stage"] == "full"]
print([(r["candidate_id"], r["combined"]) for r in full])

A stage flag indicates execution as recorded. The charts leave Holdout out because it is unobserved. Overlap between source replay batches and historical validation samples is not known.